# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/awaisxyh1234-crypto/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install duckdb -q
import duckdb

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")  # apna HF read token daalo

rel = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-06"  # jo latest sample month available hai wo use karo

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
!git clone https://github.com/awaisxyh1234-crypto/flyrank-ml-internship.git
!find flyrank-ml-internship -iname "*.csv" -o -iname "*.parquet"

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 143 (delta 55), reused 104 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.85 MiB | 6.37 MiB/s, done.
Resolving deltas: 100% (55/55), done.
flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
flyrank-ml-internship/outputs/refresh_queue_sample.csv


In [6]:
import pandas as pd
df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")  # path apna daalo
print(df.shape)
# Check for duplicates based on content_id and client_id, which are available in this dataset.
print(f"Number of duplicate rows based on (content_id, client_id): {df.duplicated(subset=['content_id', 'client_id']).sum()}")
print(f"Unique content_ids: {df['content_id'].nunique()}")
print(f"Unique client_ids: {df['client_id'].nunique()}")
# Removed date-related checks as 'date' column is not present in this dataset.

(30000, 44)
Number of duplicate rows based on (content_id, client_id): 0
Unique content_ids: 30000
Unique client_ids: 32


In [7]:
print(df.dtypes)

content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            int64
age_tier      

In [9]:
print(df.isnull().sum())
# The 'client_key' and 'date' columns are not present in this dataset.
# Group by 'client_id' to see the number of content entries per client.
print(df.groupby("client_id")["content_id"].count().sort_values(ascending=False).head(15))

content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_order                0
days_sin

In [11]:
print(df.groupby("client_id")["content_id"].count().sort_values(ascending=False).head(15))

client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
client_d029fa3a95     952
client_349c41201b     763
client_e629fa6598     720
client_e29c9c180c     708
client_2c624232cd     649
Name: content_id, dtype: int64


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: clicks, impressions, ctr, avg_position, ga4_sessions
Label: (agar prediction task nahi hai to likho "not applicable — this is a descriptive contract")
Context: client_key, page_key, query_key, date, month
Excluded: raw query text, client/domain identifiers — excluded because data-use terms allow no re-identification and no client-identifying output.

In [14]:
print(df.dtypes)

content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            int64
age_tier      

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain confirmed unique above. Below: missing-value counts per key metric, and per-client history depth to check panel balance.

In [17]:
# missing values
print(df.isnull().sum())

# per-client history depth
print(df.groupby("client_id")["content_id"].count().sort_values(ascending=False).head(15))

content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_order                0
days_sin

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Panel is unbalanced — clients have different gsc_data_start / ga4_data_start, so early history has GSC-only rows.
Sample is one anonymized month slice, not full 79M-row history — trends here are directional, not conclusive.
No causal claims possible from this data alone — only observed/measured associations.

In [19]:
print(df.groupby("client_id")["content_id"].count().sort_values(ascending=False).head(15))

client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
client_d029fa3a95     952
client_349c41201b     763
client_e629fa6598     720
client_e29c9c180c     708
client_2c624232cd     649
Name: content_id, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.